# EEG-Based Neurodevelopmental Age Estimation## Full Reproducible PipelineThis notebook runs the entire analysis pipeline end-to-end.Running all cells reproduces every result, metric, and figure.**Requirements**: `pip install -r requirements.txt`**Expected runtime**: ~7-10 minutes on a modern machine.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import logging

# Setup logging for notebook
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
                    datefmt='%H:%M:%S', stream=sys.stdout)

PROJECT_DIR = os.path.dirname(os.path.abspath('.'))
if 'neurodevelopment_project' not in PROJECT_DIR:
    PROJECT_DIR = os.path.join(PROJECT_DIR, 'neurodevelopment_project')
if not os.path.exists(os.path.join(PROJECT_DIR, 'src')):
    PROJECT_DIR = '.'
sys.path.insert(0, PROJECT_DIR)
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
for d in ['metrics', 'predictions', 'tables', 'figures', 'reports', 'models']:
    os.makedirs(os.path.join(RESULTS_DIR, d), exist_ok=True)
print(f"Project: {PROJECT_DIR}")
print(f"Results: {RESULTS_DIR}")


---## Step 1: Load and Prepare Data

In [ ]:
from src.data_loader import build_dataset
data = build_dataset()
print(f"\nSubjects: {data['n_subjects']}")
print(f"EEG features: {data['X_eeg'].shape[1]}")
print(f"Age range: {data['targets']['age'].min():.1f}–{data['targets']['age'].max():.1f}")
print(f"seqLearning partial credit available: {(~np.isnan(data['targets']['seqLearning_partial'])).sum()}")
print(f"symbolSearch accuracy available: {(~np.isnan(data['targets']['symbolSearch_accuracy'])).sum()}")


---## Step 2: Experiment 1 — EEG Age Prediction

In [ ]:
from src.age_prediction import run_age_prediction
exp1_results = run_age_prediction(data, RESULTS_DIR)


In [ ]:
# Display key results
print("\n=== Age Prediction Summary ===")
display(exp1_results['summary_df'][['model', 'cv_type', 'R2', 'MAE', 'RMSE', 'Pearson_r']].round(4))


---## Step 3: Experiment 2 — Brain-Age Gap Analysis

In [ ]:
from src.brain_age_analysis import run_brain_age_analysis
exp2_results = run_brain_age_analysis(data, exp1_results, RESULTS_DIR)


In [ ]:
print(f"\nAge-bias correlation: {exp2_results['age_bias_r_before']:.4f} → {exp2_results['age_bias_r_after']:.4f}")
display(exp2_results['association_results'][['target', 'gap_type', 'r', 'p', 'p_bonferroni', 'n']].round(4))


---## Step 4: Experiment 3 — Feature Importance*Note: Permutation importance takes ~5 minutes.*

In [ ]:
from src.interpretability import run_interpretability
exp3_results = run_interpretability(data, exp1_results, RESULTS_DIR)


In [ ]:
print("\nTop 10 by Gini importance:")
display(exp3_results['gini_importance'][['feature', 'gini_importance', 'family']].head(10).round(5))
print("\nTop 10 by permutation importance:")
display(exp3_results['permutation_importance'][['feature', 'importance_mean', 'family']].head(10).round(5))


---## Step 5: Experiment 4 — Cognitive Prediction

In [ ]:
from src.cognitive_prediction import run_cognitive_prediction
exp4_results = run_cognitive_prediction(data, RESULTS_DIR)


In [ ]:
if len(exp4_results['metrics']) > 0:
    display(exp4_results['metrics'][['target', 'feature_set', 'model', 'R2', 'MAE', 'n']].round(4))


---## Step 6: Experiment 5 — Developmental Analysis

In [ ]:
from src.developmental_analysis import run_developmental_analysis
exp5_results = run_developmental_analysis(data, exp2_results, RESULTS_DIR)


In [ ]:
print("\nEEG age trends by family:")
display(exp5_results['age_trends_family'].round(4))
print("\nGap-cognitive partial correlations:")
display(exp5_results['gap_cognitive'].round(4))


---## Step 7: Generate Figures

---
## Step 7: Experiment 6 — Age- and Sex-Specific Brain-Age Reference Analysis

Builds a reference distribution of brain-age gaps stratified by age group and sex.
For each participant, computes a **leave-one-out (LOO) reference** — the expected
gap from peers of the same age and sex, excluding that participant's own observation.

Three exploratory categories are then assigned based on how far each participant's
gap deviates from their reference.

**Key properties:**
- Uses data-driven threshold (±0.5 SD), not an arbitrary clinical cutoff
- Leave-one-out design avoids circular reference comparisons
- Sparse groups (n < 15) are flagged; their statistics are not presented as reliable
- No models are retrained — OOF RF predictions from Experiment 1 are reused
- Categories are exploratory and descriptive, **not clinical diagnoses**

In [ ]:
from src.age_sex_reference_analysis import run_age_sex_reference_analysis
exp6_results = run_age_sex_reference_analysis(data, exp2_results, RESULTS_DIR)


In [ ]:
print("=== Age-Sex Reference Statistics ===")
display(exp6_results['ref_stats'].round(3))

print("\n=== LOO Individual Stats (sample) ===")
display(exp6_results['gap_df_ref'][['subject_id','actual_age','sex','age_bin',
    'brain_age_gap','loo_ref_mean','ref_adjusted_gap','z_score',
    'percentile_within_group','reliable','category']].head(10).round(3))

print("\n=== Category Counts ===")
display(exp6_results['category_stats'].round(2))

print("\n=== Omnibus ANCOVA ===")
display(exp6_results['omnibus_df'][['outcome_label','F','p','p_bonferroni','eta2_partial','n']].round(4))

print("\n=== Sensitivity Analysis ===")
display(exp6_results['sensitivity_df'].round(3))


In [ ]:
from src.visualization import generate_all_figures
generate_all_figures(exp1_results, exp2_results, exp3_results, exp4_results, exp5_results, RESULTS_DIR)
print("All figures saved to results/figures/")


### View Key Figures

In [ ]:
from IPython.display import Image, display
fig_dir = os.path.join(RESULTS_DIR, 'figures')
for fname in sorted(os.listdir(fig_dir)):
    if fname.endswith('.png'):
        print(f"\n--- {fname} ---")
        display(Image(os.path.join(fig_dir, fname)))


---## Pipeline CompleteAll results have been saved to:- `results/metrics/` — numerical performance metrics- `results/predictions/` — per-subject predictions- `results/tables/` — summary tables- `results/figures/` — publication-quality figures- `results/reports/` — text reports- `results/models/` — trained model artifacts